In [1]:
import os
import sys
sys.path.append(os.path.join(os.getcwd(), '../'))

import optuna
import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime
from scipy.stats import linregress

from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

import lightgbm as lgb
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, precision_score, average_precision_score

## Crawl data

In [4]:
from vnstock import Vnstock
from vnstock.explorer.vci.listing import Listing

symbols_by_industries = Listing().symbols_by_industries()
symbols_by_exchange = Listing().symbols_by_exchange()
symbols_info = symbols_by_industries.merge(symbols_by_exchange)

In [5]:
symbols_list = symbols_info['symbol'].tolist()
data_count_save_path = '/remote/vast0/share-mv/tien/project/vnstock/data/data_count.csv'
run_data_dir = "/remote/vast0/share-mv/tien/project/vnstock/run_data"

In [ ]:
keep_symbols = []
data_count = []
all_raw_data = {}
for symbol in tqdm(symbols_list):
    save_path = f'/remote/vast0/share-mv/tien/project/vnstock/data/stocks/{symbol}.csv'
    if os.path.exists(save_path):
        df = pd.read_csv(save_path)
    else:
        try:
            vn_stock = Vnstock()
            stock = vn_stock.stock(symbol=symbol, source='VCI')
            df = stock.quote.history(start='2020-01-01', end='2025-02-25')
            df.to_csv(f'/remote/vast0/share-mv/tien/project/vnstock/data/stocks/{symbol}.csv', index=False)
        except Exception as e:
            print(e)
            continue
    df['time'] = pd.to_datetime(df['time'])
    df = df.sort_values('time')
    df['symbol'] = symbol
    df['next_1d_open'] = df['open'].shift(-1)
    upper_rate = 1.05
    if ((df['volume']*df['close']).rolling(10).mean()).values[-1].tolist() > 1e6 and (df['volume'] == 0).sum() < 10 and df.shape[0] > 800:
        data_count.append([symbol, df.shape[0]])
        all_raw_data[symbol] = df
        keep_symbols.append(symbol)
    
all_raw_data_df = pd.concat(list(all_raw_data.values())).reset_index(drop=True)

 38%|███▊      | 604/1591 [00:02<00:08, 118.28it/s]

Expecting value: line 1 column 1 (char 0)


 64%|██████▍   | 1023/1591 [00:05<00:04, 118.79it/s]

Expecting value: line 1 column 1 (char 0)


2025-03-24 17:46:24 - vnstock.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS
 66%|██████▌   | 1046/1591 [00:06<00:11, 48.69it/s] 

Không tìm thấy dữ liệu. Vui lòng kiểm tra lại mã chứng khoán hoặc thời gian truy xuất.


100%|██████████| 1591/1591 [00:08<00:00, 191.76it/s]


## Data Validation

Test if the current features are mapped with all_df

In [ ]:
all_df = pd.read_csv('/remote/vast0/share-mv/tien/project/vnstock/notebooks/all_df.csv')
all_df['time'] = pd.to_datetime(all_df['time'])

all_current_features = []
for f in os.listdir(run_data_dir):
    if f.startswith('current_features_'):
        df = pd.read_csv(f'{run_data_dir}/{f}')
        df['time'] = f.split('_')[-1].split('.')[0]
        df['time'] = pd.to_datetime(df['time'])
        all_current_features.append(df)

all_current_features = pd.concat(all_current_features).reset_index(drop=True)
all_cols = all_current_features.columns.tolist()
merged_df = all_current_features #.merge(all_df, on=['symbol', 'time'])

all_org_dfs = []
for f in os.listdir(run_data_dir):
    if f.startswith('org_df_'):
        df = pd.read_csv(f'{run_data_dir}/{f}')
        df['time'] = f.split('_')[-1].split('.')[0]
        df['time'] = pd.to_datetime(df['time'])
        all_org_dfs.append(df)

all_org_dfs = pd.concat(all_org_dfs).reset_index(drop=True)
cols = ['open', 'high', 'low', 'close', 'volume', 'symbol', 'time', 'next_close', 'next_min_close', 'prediction', 'close_open_diff', 'close_slope40', 'buy']
all_org_dfs = all_org_dfs[cols]
merged_df = merged_df.merge(all_org_dfs[['symbol', 'time', 'prediction']], on=['symbol', 'time'])

for c in all_cols:
    if f'{c}_x' in merged_df.columns and f'{c}_y' in merged_df.columns:
        diff_res = (merged_df[f'{c}_x'] - merged_df[f'{c}_y']).sum()
        if diff_res > 1e-5:
            print(c, diff_res)

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


In [ ]:
merged_df = merged_df.merge(all_raw_data_df[['time', 'symbol', 'next_1d_open', 'open', 'high', 'low', 'close', 'volume']], on=['time', 'symbol'], how='right')

In [39]:
ALL_SYMBOLS = all_df['symbol'].unique().tolist()

## Predictions Analysis

In [ ]:
min_time, max_time = merged_df[~merged_df['prediction'].isna()]['time'].min(), merged_df['time'].max()

In [ ]:
# from datetime import timedelta

# cur_time = min_time
# profits = []
# precisions = []
# all_counts = []
# top_symbols_scores = {}
# top_symbols_scores_count = {}
# all_top_symbols_scores_dfs = []
# while cur_time <= max_time:
#     cur_df = merged_df[merged_df['time'] == cur_time]
#     cur_df = cur_df.sort_values('prediction', ascending=False).reset_index(drop=True)
#     cur_time += timedelta(days=1)
    
#     if cur_df.shape[0] == 0:
#         continue

#     cur_df.loc[:, 'buy'] = False
#     # if self.best_threshold is not None:
#     cur_df.loc[:, 'buy'] = (cur_df['prediction'] > prediction_threshold) & (cur_df['close_open_diff_y'] > -0.05) # & (cur_df['close_slope10_y'] > 0.1)# & (cur_df['close_slope10_y'].abs() < 0.3) & (cur_df['close_open_diff_y'] > -0.05) # & (cur_df['close_slope10_y'] > 0.05) & (cur_df['close_slope40_y'] < 0.6)

#     buy_df = cur_df[cur_df['buy']]
#     buy_df = buy_df.sort_values(by='prediction', ascending=False)

#     for symbol in top_symbols_scores.keys():
#         if symbol not in buy_df['symbol'].values:
#             top_symbols_scores[symbol] = top_symbols_scores[symbol] * 0.5
    
#     for _, row in buy_df.iterrows():
#         symbol = row['symbol']
#         if symbol in top_symbols_scores:
#             top_symbols_scores[symbol] = (top_symbols_scores[symbol] + row['prediction'])/2
#         else:
#             top_symbols_scores[symbol] = row['prediction']
#         top_symbols_scores_count[symbol] = top_symbols_scores_count.get(symbol, 0) + 1

#         if len(top_symbols_scores) > num_top_symbols:
#             break

#     all_symbols = list(top_symbols_scores.keys())
#     for symbol in all_symbols:
#         if top_symbols_scores[symbol] < prediction_threshold:
#             del top_symbols_scores[symbol]

#     orders = []
#     cur_order_count = 0
#     top_symbols_scores_count_df = pd.DataFrame(top_symbols_scores_count.items(), columns=['symbol', 'count'])
#     top_symbols_scores_df = pd.DataFrame(top_symbols_scores.items(), columns=['symbol', 'score'])
#     top_symbols_scores_df = top_symbols_scores_df.merge(cur_df[['symbol', 'close', 'label_y', 'next_close', 'buy', 'next_min_close', 'next_1d_open']], on='symbol', how='left')
#     top_symbols_scores_df = top_symbols_scores_df.merge(top_symbols_scores_count_df, on='symbol', how='left')
#     top_symbols_scores_df = top_symbols_scores_df.sort_values(by='score', ascending=False)
#     top_symbols_scores_df['next_close_diff'] = top_symbols_scores_df['next_close']/top_symbols_scores_df['next_1d_open']-1
#     # print(cur_time, top_symbols_scores_df[['symbol', 'score', 'count']])
#     all_top_symbols_scores_dfs.append(top_symbols_scores_df)
    
#     if top_symbols_scores_df[top_symbols_scores_df['count'] >= count_threshold].shape[0] > 0:
#         a = top_symbols_scores_df[(top_symbols_scores_df['label_y'] == True) & (top_symbols_scores_df['count'] >= count_threshold)].shape[0]
#         b = top_symbols_scores_df[top_symbols_scores_df['count'] >= count_threshold].shape[0]
#         precisions.append((cur_time, a, b, a/b))
#         profits.append(((top_symbols_scores_df[(top_symbols_scores_df['count'] >= count_threshold) & (top_symbols_scores_df['next_min_close']/top_symbols_scores_df['next_1d_open']>0.95)]['next_close_diff']).sum()+(top_symbols_scores_df[(top_symbols_scores_df['count'] >= count_threshold) & (top_symbols_scores_df['next_min_close']/top_symbols_scores_df['next_1d_open']<0.95)].shape[0]*-0.06))/top_symbols_scores_df[top_symbols_scores_df['count'] >= count_threshold].shape[0])
#         print(cur_time, top_symbols_scores_df[top_symbols_scores_df['count'] >= count_threshold])
        
#         if a == 0:
#             del top_symbols_scores_df['buy']

In [50]:
def convert_df_to_dict(df):
    global ALL_SYMBOLS
    dct = {}
    for symbol in ALL_SYMBOLS:
        symbol_df = df[df['symbol'] == symbol].reset_index(drop=True)
        dct[symbol] = symbol_df
    return dct

In [220]:
from datetime import timedelta

from trade_bot.account_manager import AccountManager, Order

max_orders = 4
num_top_symbols = 10
count_threshold = 2
prediction_threshold = 0.55
cutloss_rate = 0.05
take_profit_delta_time = 10

cur_time = min_time - timedelta(days=1)
profits = []
precisions = []
all_counts = []
top_symbols_scores = {}
top_symbols_scores_count = {}
all_top_symbols_scores_dfs = []

initial_balance = 1e5
account_manager = AccountManager(initial_balance)

start_time = None
while cur_time <= max_time:
    cur_time += timedelta(days=1)
    if start_time is None:
        print(f'Setting start time as {cur_time} in LGBMStrategy')
        start_time = cur_time

    cur_df = merged_df[merged_df['time'] == cur_time]
    cur_df = cur_df.sort_values('prediction', ascending=False).reset_index(drop=True)
    cur_dict = convert_df_to_dict(cur_df)
    
    account_manager.execute_orders(current_time=cur_time, price_data=cur_dict)

    active_orders = account_manager.get_active_orders()
    active_symbols = [order.symbol for order in active_orders]
    money_balance = account_manager.get_money_balance()
    all_balance = account_manager.get_all_balance()
    print('All balance:', all_balance)

    for order in active_orders:
        if order.status == 'no_price_data':
            order.buy_time += timedelta(days=1)
            order.take_profit_time += timedelta(days=1)
    
    orders = []
    if cur_df.shape[0] == 0:
        continue
    
    active_orders_length = len(active_orders)
    for order in active_orders:
        if order.status == 'triggered' and order.direction == 'buy':
            data = cur_df[cur_df['symbol'] == order.symbol]
            symbol = order.symbol
            volume = order.volume
            buy_price = order.buy_price
            if data.shape[0] == 0:
                continue
            
            if cur_time - timedelta(days=2) >= order.buy_time:
                close_price = data.iloc[-1]['close']
                if close_price * (1 + order.cutloss_rate) < buy_price:
                    order.take_profit_time = cur_time
                    order.type = 'open'
                    order.note = 'cutloss'
                    continue

                # Check for sell signals
                if close_price > order.buy_price * (1 + order.take_profit_rate):
                    order.take_profit_rate = (close_price / order.buy_price) - 1.03
                    order.trigger_take_profit = True
                    order.note = 'change take profit rate'
                    order.take_profit_time = order.take_profit_time + timedelta(days=take_profit_delta_time) # min(order.take_profit_time + timedelta(days=10), order.buy_time + timedelta(days=20))
                    continue

                if order.trigger_take_profit and close_price < order.buy_price * (1 + order.take_profit_rate):
                    order.note = f'sell for profit {order.take_profit_rate}'
                    order.take_profit_time = cur_time
                    order.type = 'open'
                
                # sell_signals_df = current_features[(current_features['cur_volume_vs_avg10_diff'] < -0.4) & (current_features['prev1d_cur_volume_vs_avg10_diff'] < -0.4) & (current_features['prev2d_cur_volume_vs_avg10_diff'] < -0.4)]
                # if sell_signals_df[sell_signals_df['symbol'] == symbol].shape[0] > 0:
                #     order.note = 'sell because of volume signals'
                #     order.take_profit_time = cur_time
                #     order.type = 'open'

    # Strategy: Buy top max_orders stocks with the highest prediction
    if not (active_orders_length > max_orders or money_balance < all_balance/max_orders) and cur_df.shape[0] > 0:
        org_df = cur_df.copy()
        org_df.loc[:, 'buy'] = False
        # if best_threshold is not None:
        org_df.loc[:, 'buy'] = (org_df['prediction'] > prediction_threshold) & (org_df['close_slope40'] > 0.) & (org_df['close_slope40'] < 0.3) & (org_df['close_slope10'] < 0.05) & (org_df['close_slope10'] > 0.)

        buy_df = org_df[org_df['buy']]
        buy_df = buy_df.sort_values(by='prediction', ascending=False)

        for symbol in top_symbols_scores.keys():
            if symbol not in buy_df['symbol'].values:
                top_symbols_scores[symbol] = top_symbols_scores[symbol] * 0.5
        
        for _, row in buy_df.iterrows():
            symbol = row['symbol']
            if symbol in top_symbols_scores:
                top_symbols_scores[symbol] = (top_symbols_scores[symbol] + row['prediction'])/2
            else:
                top_symbols_scores[symbol] = row['prediction']
            top_symbols_scores_count[symbol] = top_symbols_scores_count.get(symbol, 0) + 1

            if len(top_symbols_scores) > num_top_symbols:
                break

        all_symbols = list(top_symbols_scores.keys())
        for symbol in all_symbols:
            if top_symbols_scores[symbol] < prediction_threshold:
                del top_symbols_scores[symbol]
        
        cur_order_count = 0
        top_symbols_scores_count_df = pd.DataFrame(top_symbols_scores_count.items(), columns=['symbol', 'count'])
        top_symbols_scores_df = pd.DataFrame(top_symbols_scores.items(), columns=['symbol', 'score'])
        top_symbols_scores_df = top_symbols_scores_df.merge(cur_df[['symbol', 'close']], on='symbol', how='left')
        top_symbols_scores_df = top_symbols_scores_df.merge(top_symbols_scores_count_df, on='symbol', how='left')
        top_symbols_scores_df = top_symbols_scores_df.sort_values(by='score', ascending=False)
        for _, row in top_symbols_scores_df.iterrows():
            symbol = row['symbol']
            count = row['count']
            if count < count_threshold or bool(np.isnan(row['close'])):
                continue
            volume = float(all_balance/max_orders/row['close'])
            volume -= volume % 100
            if volume == 0:
                continue
            if symbol in active_symbols:
                continue
            orders.append(Order(
                symbol = symbol,
                volume = volume,
                direction = 'buy',
                type = 'open',
                cutloss_rate=cutloss_rate,
                buy_time = cur_time + timedelta(days=1),
                take_profit_time = cur_time + timedelta(days=take_profit_delta_time))
            )
            cur_order_count += 1
            active_orders_length += 1
            if active_orders_length >= 4:
                break

    account_manager.add_orders(orders=orders)
    account_manager.report(current_time=cur_time, price_data=cur_dict)

Setting start time as 2024-06-03 00:00:00 in LGBMStrategy
All balance: 100000.0
[2024-06-03 00:00:00]	All balance: 100000.0
			Money balance: 100000.0
			Active orders: []
All balance: 100000.0
[2024-06-04 00:00:00]	All balance: 100000.0
			Money balance: 100000.0
			Active orders: []
All balance: 100000.0
[2024-06-05 00:00:00]	All balance: 100000.0
			Money balance: 100000.0
			Active orders: []
All balance: 100000.0
[2024-06-06 00:00:00]	All balance: 100000.0
			Money balance: 100000.0
			Active orders: []
All balance: 100000.0
[2024-06-07 00:00:00]	All balance: 100000.0
			Money balance: 100000.0
			Active orders: []
All balance: 100000.0
All balance: 100000.0
All balance: 100000.0
[2024-06-10 00:00:00]	All balance: 100000.0
			Money balance: 100000.0
			Active orders: [Order(id=1195, symbol='DXS', volume=3000.0, direction='buy', type='open', take_profit_rate=0.05, trigger_take_profit=False, cutloss_rate=0.05, buy_time=Timestamp('2024-06-11 00:00:00'), take_profit_time=Timestamp('20

In [219]:
cutloss_orders = []
for orders in account_manager.trackings['take_profit_orders']:
    for order in orders:
        if order.sell_price / order.buy_price < 1:
            cutloss_orders.append((order, order.sell_price / order.buy_price))

sorted(cutloss_orders, key=lambda x: x[1])

[(Order(id=1193, symbol='SCR', volume=4500.0, direction='sell', type='close', take_profit_rate=0.05, trigger_take_profit=False, cutloss_rate=0.05, buy_time=Timestamp('2025-01-03 00:00:00'), take_profit_time=Timestamp('2025-01-06 00:00:00'), status='triggered', buy_cost=np.float64(25876.494000000002), sell_cost=np.float64(23357.88), buy_price=np.float64(5.74), sell_price=np.float64(5.2), note='cutloss'),
  np.float64(0.9059233449477352))]

In [ ]:
[order for order in account_manager.trackings['take_profit_orders'] if order.note == 'take_profit']

In [89]:
cur_time

Timestamp('2024-08-24 00:00:00')

In [144]:
sum([p[1] for p in precisions]),sum([p[2] for p in precisions]),sum([p[1] for p in precisions])/sum([p[2] for p in precisions])

(111, 272, 0.40808823529411764)

In [145]:
sum(profits)/len(profits)

np.float64(0.03323349748727173)

In [ ]:
from datetime import timedelta

max_ = 0
best_cfg = None
num_top_symbols = 10
count_threshold = 2
prediction_threshold = 0.55
top_symbols_scores_count = {}

for close_open_diff_y_thres in [-0.01, -0.015, -0.02, -0.025, -0.03, -0.035, -0.04, -0.045, -0.05, -0.055, -0.06]:
    for close_slope10_y_thres in range(1, 50, 1):
        close_slope10_y_thres /= 50

        min_time, max_time = merged_df['time'].min(), merged_df['time'].max()

        cur_time = min_time
        profits = []
        precisions = []
        all_counts = []
        top_symbols_scores = {}
        top_symbols_scores_count = {}
        all_top_symbols_scores_dfs = []
        while cur_time <= max_time:
            cur_df = merged_df[merged_df['time'] == cur_time]
            cur_df = cur_df.sort_values('prediction', ascending=False).reset_index(drop=True)
            cur_time += timedelta(days=1)
            
            if cur_df.shape[0] == 0:
                continue

            cur_df.loc[:, 'buy'] = False
            # if best_threshold is not None:
            cur_df.loc[:, 'buy'] = (cur_df['prediction'] > prediction_threshold) & (cur_df['close_open_diff_y'] > close_open_diff_y_thres) & (cur_df['close_slope10_y'] > close_slope10_y_thres)# & (cur_df['close_slope10_y'].abs() < 0.3) & (cur_df['close_open_diff_y'] > -0.05) # & (cur_df['close_slope10_y'] > 0.05) & (cur_df['close_slope40_y'] < 0.6)

            buy_df = cur_df[cur_df['buy']]
            buy_df = buy_df.sort_values(by='prediction', ascending=False)

            for symbol in top_symbols_scores.keys():
                if symbol not in buy_df['symbol'].values:
                    top_symbols_scores[symbol] = top_symbols_scores[symbol] * 0.5
            
            for _, row in buy_df.iterrows():
                symbol = row['symbol']
                if symbol in top_symbols_scores:
                    top_symbols_scores[symbol] = (top_symbols_scores[symbol] + row['prediction'])/2
                else:
                    top_symbols_scores[symbol] = row['prediction']
                top_symbols_scores_count[symbol] = top_symbols_scores_count.get(symbol, 0) + 1

                if len(top_symbols_scores) > num_top_symbols:
                    break

            all_symbols = list(top_symbols_scores.keys())
            for symbol in all_symbols:
                if top_symbols_scores[symbol] < prediction_threshold:
                    del top_symbols_scores[symbol]

            orders = []
            cur_order_count = 0
            top_symbols_scores_count_df = pd.DataFrame(top_symbols_scores_count.items(), columns=['symbol', 'count'])
            top_symbols_scores_df = pd.DataFrame(top_symbols_scores.items(), columns=['symbol', 'score'])
            top_symbols_scores_df = top_symbols_scores_df.merge(cur_df[['symbol', 'close', 'label_y', 'next_close', 'buy', 'next_min_close', 'next_1d_open']], on='symbol', how='left')
            top_symbols_scores_df = top_symbols_scores_df.merge(top_symbols_scores_count_df, on='symbol', how='left')
            top_symbols_scores_df = top_symbols_scores_df.sort_values(by='score', ascending=False)
            top_symbols_scores_df['next_close_diff'] = top_symbols_scores_df['next_close']/top_symbols_scores_df['next_1d_open']-1
            all_top_symbols_scores_dfs.append(top_symbols_scores_df)
            
            if top_symbols_scores_df[top_symbols_scores_df['count'] >= count_threshold].shape[0] > 0:
                a = top_symbols_scores_df[(top_symbols_scores_df['label_y'] == True) & (top_symbols_scores_df['count'] >= count_threshold)].shape[0]
                b = top_symbols_scores_df[top_symbols_scores_df['count'] >= count_threshold].shape[0]
                precisions.append((cur_time, a, b, a/b))
                profits.append(((top_symbols_scores_df[(top_symbols_scores_df['count'] >= count_threshold) & (top_symbols_scores_df['next_min_close']/top_symbols_scores_df['next_1d_open']>0.95)]['next_close_diff']).sum()+(top_symbols_scores_df[(top_symbols_scores_df['count'] >= count_threshold) & (top_symbols_scores_df['next_min_close']/top_symbols_scores_df['next_1d_open']<0.95)].shape[0]*-0.06))/top_symbols_scores_df[top_symbols_scores_df['count'] >= count_threshold].shape[0])
                
                if a == 0:
                    del top_symbols_scores_df['buy']

        if len(profits) > 20 and sum(profits)/len(profits) > max_:
            best_cfg = (close_open_diff_y_thres, close_slope10_y_thres)
            max_ = sum(profits)/len(profits)
            print(f'current best cfg: {best_cfg}, max_val = {max_}')

current best cfg: (-0.01, 0.02), max_val = 0.026125590850790196
current best cfg: (-0.01, 0.04), max_val = 0.03535310151568257
current best cfg: (-0.01, 0.06), max_val = 0.04070982112256427
current best cfg: (-0.015, 0.14), max_val = 0.05325567696481575
current best cfg: (-0.02, 0.14), max_val = 0.054796111172922324
current best cfg: (-0.035, 0.14), max_val = 0.057457745923224376
current best cfg: (-0.04, 0.2), max_val = 0.05892469785650783
current best cfg: (-0.05, 0.14), max_val = 0.07138904217518445
current best cfg: (-0.05, 0.2), max_val = 0.07670916510432212
